---

# Dudley Economic Activity — Data Cleaning

## Objectives
- Inspect the dataset structure and geographical coverage.
- Check for missing values, duplicates and incorrect data types.
- Prepare the economic activity data for analysis.
- Save a cleaned copy while preserving the raw dataset.

## Inputs
- `raw_datasets/dudley_economic_activity.csv`

## Outputs

- Cleaned dataset: `cleaned_datasets/dudley_economic_activity_cleaned.csv`
- Documented data quality checks and cleaning decisions.

## Additional Comments

- The original raw dataset will remain unchanged.
- The geographical coverage and category definitions will be
  checked before cleaning.
- Missing values and duplicates will be investigated before
  deciding how to handle them.
- Economic inactivity and unemployment are different measures
  and will be kept distinct during preparation and analysis.

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [4]:
if os.path.basename(os.getcwd()) == "jupyter_notebooks":
    os.chdir("..")

print(os.getcwd())

c:\Dudley employment barriers


Confirm the new current directory

In [5]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

## 1. Import Libraries

- os: manage the working directory and output folders.
- pandas: load, inspect, clean and save the dataset.

In [6]:
import os
import pandas as pd

---

## 2. Inspect the Raw Dataset

Check the first ten lines to identify column headings and any
introductory text before loading the CSV into pandas.

In [7]:
raw_path = "raw_datasets/dudley_economic_activity.csv"

with open(raw_path, encoding="utf-8-sig") as file:
    for _ in range(10):
        line = file.readline()
        if not line:
            break
        print(line.rstrip())


"TS066 - Economic activity status"
"ONS Crown Copyright Reserved [from Nomis on 14 October 2026]"
"Population :","All usual residents aged 16 years and over"
"Units      :","Persons"
"Date       :","2021"

"2021 super output area - lower layer","Total: All usual residents aged 16 years and over","Economically active (excluding full-time students)","Economically active (excluding full-time students):In employment","Economically active (excluding full-time students): Unemployed","Economically inactive: Student","Economically inactive: Looking after home or family","Economically inactive: Long-term sick or disabled","Economically inactive: Other"

"E01009763 : Dudley 001A",1637,970,888,82,97,164,73,70


### Findings from the Raw File Inspection

- The dataset is TS066: Economic activity status, from Census 2021.
- It covers all usual residents aged 16 years and over.
- Values represent counts of persons.
- The geography is 2021 Lower Layer Super Output Areas (LSOAs).
- The first six lines contain metadata and a blank line.
  These will be skipped when loading the dataset.
- The column headings appear on the seventh line.
- The first visible data record is Dudley 001A.

### 2.1 Load the Dataset

Load the CSV into a pandas DataFrame, skipping the six introductory
lines so the column headings are read correctly.

Display the dataset dimensions and the first and last five rows
to check the structure and identify any footer notes.

In [11]:
# Load the CSV, skipping the six introductory lines.
df = pd.read_csv(
    raw_path,
    skiprows=6,
    encoding="utf-8-sig"
)

# Show the dataset dimensions.
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Display all rows and columns without pandas truncation.
print(df.to_string(index=True))

Rows: 208, Columns: 9
    2021 super output area - lower layer                          Total: All usual residents aged 16 years and over  Economically active (excluding full-time students)  Economically active (excluding full-time students):In employment  Economically active (excluding full-time students): Unemployed  Economically inactive: Student  Economically inactive: Looking after home or family  Economically inactive: Long-term sick or disabled  Economically inactive: Other
0                E01009763 : Dudley 001A                                                                       1637                                               970.0                                                             888.0                                                            82.0                            97.0                                                164.0                                               73.0                          70.0
1                E01009764 : Dudley 001B          

Section 2 content

### Findings from Loading and Inspecting the Full Dataset

- The imported dataset contains 208 rows and 9 columns.
- Rows 0–202 contain 203 Dudley LSOA records.
- Rows 203–206 contain disclosure-control notes.
- Row 207 is empty.
- Area codes and area names are combined in the first column.
- No missing values are visible within the 203 area records;
  this will be confirmed with a missing-value check.
- Count values are displayed with decimal places in several
  columns. Data types will be checked after removing non-data rows.
- The selected columns include an economically active subtotal
  and its employment and unemployment components. These must
  not be added together, as this would double-count people.
- The export does not include every economic activity category,
  so the selected categories should not be expected to sum
  to the total population.
- The source notes describe confidentiality adjustments to counts,
  which should be considered when interpreting small values.

The next step is to remove the five non-data rows while preserving
all 203 area records and keeping the original CSV unchanged.

## 3. Remove Non-Data Rows

Remove the four footer-note rows and the empty row by retaining
records with a valid Dudley LSOA code and name.

Create a separate working DataFrame, keeping the imported data
in df unchanged. Display all excluded rows and the row counts
to verify what was removed.

In [12]:
# Developed with assistance from ChatGPT; output reviewed before proceeding.
# Identify records containing a Dudley LSOA code and name.
area_column = df.columns[0]
area_mask = df[area_column].str.fullmatch(
    r"E\d{8}\s*:\s*Dudley \d{3}[A-Z]",
    na=False
)

# Show every excluded row so the removal can be checked.
removed_rows = df.loc[~area_mask]
print("EXCLUDED ROWS:")
print(removed_rows.to_string(index=True))

# Create a working copy containing only Dudley area records.
df_clean = df.loc[area_mask].copy().reset_index(drop=True)

# Report the number of records before and after filtering.
print(f"\nOriginal rows: {len(df)}")
print(f"Removed rows: {len(removed_rows)}")
print(f"Remaining area records: {len(df_clean)}")
print(f"Columns: {df_clean.shape[1]}")

EXCLUDED ROWS:
    2021 super output area - lower layer                          Total: All usual residents aged 16 years and over  Economically active (excluding full-time students)  Economically active (excluding full-time students):In employment  Economically active (excluding full-time students): Unemployed  Economically inactive: Student  Economically inactive: Looking after home or family  Economically inactive: Long-term sick or disabled  Economically inactive: Other
203                                  NaN    In order to protect against disclosure of personal information, records                                                 NaN                                                               NaN                                                             NaN                             NaN                                                  NaN                                                NaN                           NaN
204                                  NaN  have been swapp

### Findings from Removing Non-Data Rows

- Removed four disclosure-control note rows and one empty row.
- Retained all 203 Dudley LSOA records and all 9 columns.
- The excluded rows contain no area records.
- The working DataFrame, df_clean, now contains only area data,
  with its index reset.
- The imported DataFrame, df, and the original CSV remain unchanged.
- The disclosure-control explanation remains documented above
  and preserved in the raw file.

## 4. Check Missing Values, Duplicates and Data Types

Check all 203 area records for missing values, fully duplicated
rows and repeated area identifiers.

Inspect the data type of every column before converting counts
to numeric types. Report any affected rows for investigation
before making further changes.

In [13]:
# Developed with assistance from ChatGPT.

# Summarise missing values and data types for every column.
quality_summary = pd.DataFrame({
    "Data type": df_clean.dtypes.astype(str),
    "Missing values": df_clean.isna().sum()
})

print("COLUMN CHECKS:")
print(quality_summary.to_string())

# Identify all rows involved in duplicates.
duplicate_rows = df_clean.duplicated(keep=False)
duplicate_areas = df_clean.duplicated(
    subset=[area_column],
    keep=False
)

print(f"\nRows in duplicate groups: {duplicate_rows.sum()}")
print(f"Rows with repeated area identifiers: {duplicate_areas.sum()}")

# Display every record affected by missing values or duplicates.
affected_rows = (
    df_clean.isna().any(axis=1)
    | duplicate_rows
    | duplicate_areas
)

print("\nRECORDS REQUIRING INVESTIGATION:")
if affected_rows.any():
    print(df_clean.loc[affected_rows].to_string(index=True))
else:
    print("No missing values or duplicates found.")

COLUMN CHECKS:
                                                                 Data type  Missing values
2021 super output area - lower layer                                object               0
Total: All usual residents aged 16 years and over                   object               0
Economically active (excluding full-time students)                 float64               0
Economically active (excluding full-time students):In employment   float64               0
Economically active (excluding full-time students): Unemployed     float64               0
Economically inactive: Student                                     float64               0
Economically inactive: Looking after home or family                float64               0
Economically inactive: Long-term sick or disabled                  float64               0
Economically inactive: Other                                       float64               0

Rows in duplicate groups: 0
Rows with repeated area identifiers: 0

RECORD

### Findings from Data Quality Checks

- All 203 area records have no missing values across the 9 columns.
- No fully duplicated rows or repeated area identifiers were found.
- No records need to be removed or filled following these checks.
- The area identifier column has an object data type, suitable
  for its current text values.
- The total population column also has an object data type,
  because it originally included footer text.
- The remaining seven count columns have float64 data types.
- All eight count columns will be checked for numeric, whole-number
  and non-negative values before conversion to integer types.

## 5. Validate and Convert Count Columns

Check all eight count columns for non-numeric, missing, negative
or fractional values.

If every value passes these checks, convert the count columns
to integers because they represent numbers of people.
If any values fail, display the affected records for investigation
and leave the original columns unchanged.

In [14]:
# Developed with assistance from ChatGPT.

# Select the eight count columns, excluding the area identifier.
count_columns = df_clean.columns.drop(area_column)

# Create a numeric copy for validation.
numeric_counts = df_clean[count_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Check every count for missing, non-numeric or invalid values.
invalid_counts = (
    numeric_counts.isna()
    | numeric_counts.lt(0)
    | numeric_counts.mod(1).ne(0)
)

print("INVALID VALUES PER COLUMN:")
print(invalid_counts.sum().to_string())

# Developed with assistance from ChatGPT.
# Convert only if all count values pass validation.
if invalid_counts.any().any():
    print("\nRECORDS REQUIRING INVESTIGATION:")
    print(
        df_clean.loc[invalid_counts.any(axis=1)].to_string(
            index=True
        )
    )
    print("\nNo column types were changed.")
else:
    df_clean = df_clean.astype({
        column: "int64" for column in count_columns
    })

    print("\nAll eight count columns converted to integers.")
    print("\nUPDATED DATA TYPES:")
    print(df_clean.dtypes.to_string())
    print(f"\nRows: {len(df_clean)}, Columns: {df_clean.shape[1]}")

INVALID VALUES PER COLUMN:
Total: All usual residents aged 16 years and over                   0
Economically active (excluding full-time students)                  0
Economically active (excluding full-time students):In employment    0
Economically active (excluding full-time students): Unemployed      0
Economically inactive: Student                                      0
Economically inactive: Looking after home or family                 0
Economically inactive: Long-term sick or disabled                   0
Economically inactive: Other                                        0

All eight count columns converted to integers.

UPDATED DATA TYPES:
2021 super output area - lower layer                                object
Total: All usual residents aged 16 years and over                    int64
Economically active (excluding full-time students)                   int64
Economically active (excluding full-time students):In employment     int64
Economically active (excluding full-time stu

### Findings from Count Validation and Conversion

- All eight count columns passed validation across all 203 records.
- No missing, non-numeric, negative or fractional values were found.
- All eight count columns were successfully converted to int64.
- This removes unnecessary decimal places while preserving
  the original count values.
- The area identifier column remains text.
- The dataset still contains 203 rows and 9 columns.

## 6. Separate Area Codes and Names

Split the combined area identifier into LSOA_Code and LSOA_Name.
The code will be used to join this dataset with other datasets,
once their geographical boundaries have been checked.

Keep the original identifier for now and display all 203 mappings
to verify the split.

In [15]:
# Developed with assistance from ChatGPT.

# Split on the first colon and remove surrounding whitespace.
area_parts = df_clean[area_column].str.split(
    ":", n=1, expand=True
)

df_clean["LSOA_Code"] = area_parts[0].str.strip()
df_clean["LSOA_Name"] = area_parts[1].str.strip()

# Display every original identifier alongside the separated values.
print("AREA CODE AND NAME MAPPINGS:")
print(
    df_clean[
        [area_column, "LSOA_Code", "LSOA_Name"]
    ].to_string(index=False)
)

# Developed with assistance from ChatGPT.
# Split on the first colon and remove surrounding whitespace.
area_parts = df_clean[area_column].str.split(
    ":", n=1, expand=True
)

df_clean["LSOA_Code"] = area_parts[0].str.strip()
df_clean["LSOA_Name"] = area_parts[1].str.strip()

# Display every original identifier alongside the separated values.
print("AREA CODE AND NAME MAPPINGS:")
print(
    df_clean[
        [area_column, "LSOA_Code", "LSOA_Name"]
    ].to_string(index=False)
)

# Developed with assistance from ChatGPT.
# Check the separated identifiers across all records.
print("\nIDENTIFIER CHECKS:")
print(f"Unique LSOA codes: {df_clean['LSOA_Code'].nunique()}")
print(f"Missing codes: {df_clean['LSOA_Code'].isna().sum()}")
print(f"Missing names: {df_clean['LSOA_Name'].isna().sum()}")
print(f"Duplicate codes: {df_clean['LSOA_Code'].duplicated().sum()}")
print(f"Rows: {len(df_clean)}, Columns: {df_clean.shape[1]}")
# Check the separated identifiers across all records.
print("\nIDENTIFIER CHECKS:")
print(f"Unique LSOA codes: {df_clean['LSOA_Code'].nunique()}")
print(f"Missing codes: {df_clean['LSOA_Code'].isna().sum()}")
print(f"Missing names: {df_clean['LSOA_Name'].isna().sum()}")
print(f"Duplicate codes: {df_clean['LSOA_Code'].duplicated().sum()}")
print(f"Rows: {len(df_clean)}, Columns: {df_clean.shape[1]}")

# Developed with assistance from ChatGPT.
# Split on the first colon and remove surrounding whitespace.
area_parts = df_clean[area_column].str.split(
    ":", n=1, expand=True
)

df_clean["LSOA_Code"] = area_parts[0].str.strip()
df_clean["LSOA_Name"] = area_parts[1].str.strip()

# Display every original identifier alongside the separated values.
print("AREA CODE AND NAME MAPPINGS:")
print(
    df_clean[
        [area_column, "LSOA_Code", "LSOA_Name"]
    ].to_string(index=False)
)

# Developed with assistance from ChatGPT.
# Check the separated identifiers across all records.
print("\nIDENTIFIER CHECKS:")
print(f"Unique LSOA codes: {df_clean['LSOA_Code'].nunique()}")
print(f"Missing codes: {df_clean['LSOA_Code'].isna().sum()}")
print(f"Missing names: {df_clean['LSOA_Name'].isna().sum()}")
print(f"Duplicate codes: {df_clean['LSOA_Code'].duplicated().sum()}")
print(f"Rows: {len(df_clean)}, Columns: {df_clean.shape[1]}")

AREA CODE AND NAME MAPPINGS:
2021 super output area - lower layer LSOA_Code   LSOA_Name
             E01009763 : Dudley 001A E01009763 Dudley 001A
             E01009764 : Dudley 001B E01009764 Dudley 001B
             E01009767 : Dudley 001C E01009767 Dudley 001C
             E01009769 : Dudley 001D E01009769 Dudley 001D
             E01009770 : Dudley 001E E01009770 Dudley 001E
             E01009773 : Dudley 002A E01009773 Dudley 002A
             E01009777 : Dudley 002B E01009777 Dudley 002B
             E01009778 : Dudley 002C E01009778 Dudley 002C
             E01009895 : Dudley 002D E01009895 Dudley 002D
             E01009896 : Dudley 003A E01009896 Dudley 003A
             E01009897 : Dudley 003B E01009897 Dudley 003B
             E01009898 : Dudley 003C E01009898 Dudley 003C
             E01009901 : Dudley 003D E01009901 Dudley 003D
             E01009902 : Dudley 003E E01009902 Dudley 003E
             E01009765 : Dudley 004A E01009765 Dudley 004A
             E01009766 : Du

### Findings from Separating Area Codes and Names

- Successfully separated the area identifiers into LSOA_Code
  and LSOA_Name.
- The separated values match the original identifiers.
- All 203 LSOA codes are unique.
- No missing codes, missing names or duplicate codes were found.
- The original combined identifier has been retained for now.
- The dataset contains 203 rows and 11 columns after adding
  the two new fields.
- LSOA_Code provides a joining key for other datasets, subject
  to checking that their geographical boundaries are compatible.

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
